# 01 - Compreensão dos Dados, Qualidade e Definição de Labels (Alvos)

## 1. Contextualização do Problema e Âmbito do Projeto
O presente projeto é desenvolvido no âmbito da unidade curricular de **Laboratório de Inteligência Artificial e Ciência de Dados (Lab AI & CD)** e tem como objetivo conceber e avaliar uma *pipeline* de Ciência de Dados para extrair biomarcadores de imagem (*radiomics*) e classificar as avaliações de malignidade atribuídas por radiologistas a nódulos pulmonares em imagens de Tomografia Computorizada (CT).

O cancro do pulmão continua a ser uma das principais causas de mortalidade oncológica global. O diagnóstico atempado é determinante no prognóstico: segundo os dados do programa SEER (*Surveillance, Epidemiology, and End Results*), a taxa de sobrevivência a cinco anos para lesões diagnosticadas em estádio localizado ronda os **65%**, caindo drasticamente para valores inferiores a **10%** quando a neoplasia é detetada em estádios avançados ou metastizados.

Embora a tomografia computorizada de tórax seja a ferramenta de referência para o rastreio e acompanhamento de lesões pulmonares, a avaliação visual humana sofre de uma acentuada **variabilidade intra e inter-observador** e acarreta uma elevada carga cognitiva perante o volume maciço de cortes anatómicos. A utilização de abordagens computacionais de análise quantitativa de textura e geometria (*Radiomics*) aliada a algoritmos de aprendizagem automática permite padronizar e dar suporte objetivo à tomada de decisão clínica.

---

## 2. Os Datasets: LIDC-IDRI e LNDb
Para mitigar o sobreajuste (*overfitting*) a uma única população/scanner e assegurar a robustez e generalização dos modelos, o projeto recorre a dois conjuntos de dados complementares:

* **[LIDC-IDRI](https://doi.org/10.7937/K9/TCIA.2015.LO9QL9SX)** *(The Cancer Imaging Archive)*: Base de dados pública de referência internacional ([Armato et al., 2011](https://doi.org/10.1118/1.3528204)), contendo exames de CT de **1010 pacientes** (1018 séries de CT). O processo de anotação decorreu em duas fases (*blinded* seguida de revisão *unblinded*) por quatro radiologistas especialistas, categorizando os achados em:
  * Nódulos $\ge 3\text{ mm}$ (delineados corte a corte com contornos espaciais e notas de textura/malignidade de 1 a 5);
  * Nódulos $< 3\text{ mm}$;
  * Não-nódulos $\ge 3\text{ mm}$.
* **[LNDb](https://doi.org/10.5281/zenodo.6613714)** *(Centro Hospitalar e Universitário de São João - CHUSJ, Porto)*: Conjunto de dados retrospetivo independente ([Pedrosa et al., 2019, 2021](https://doi.org/10.1016/j.media.2021.102027)) com **294 exames de CT**. Ao contrário do protocolo colaborativo do LIDC-IDRI, o LNDb seguiu um protocolo de leitura cega individual (*single-blinded*) efetuado por 5 radiologistas, refletindo a variabilidade e o ruído característicos de um fluxo de trabalho hospitalar real.

---

## 3. Resumo, Objetivo do Notebook & Decisões Principais

O objetivo deste primeiro notebook é validar geometricamente as anotações médicas contra os dados em bruto e consolidar os alvos (*ground truth labels*) de treino para nódulos pulmonares $\ge 3\text{ mm}$.

**Decisões e Princípios Metodológicos:**
* **Fontes Intactas (*Read-Only*):** Os ficheiros originais de imagens e metadados permanecem estritamente intactos para assegurar a total reprodutibilidade da *pipeline*.
* **Tabelas de Suporte Não Utilizadas:** As tabelas de suporte do LIDC-IDRI (metadados das séries, contagens de nódulos e diagnósticos) não são usadas. Os identificadores de paciente e de série provêm do `pylidc` e os campos clínicos e demográficos não entram como variáveis preditoras, o que evita imputar valores em falta mas não garante, por si só, a ausência de enviesamentos.
* **Natureza Subjetiva dos Alvos (*Targets*):** As pontuações de malignidade (escala 1 a 5) refletem a perceção radiológica e probabilidade subjetiva atribuída pelos médicos, e **não** um diagnóstico patológico/histológico definitivo (que se encontra omisso na vasta maioria dos pacientes).
* **Consenso e Resolução de Conflitos:** O alvo numérico de treino resulta da média arredondada (`floor(mean + 0.5)`). Casos de elevada discórdia médica (amplitude $\ge 3$ com concordância de consenso $< 2/3$) ou agrupamentos ambíguos do mesmo leitor são congelados para **revisão humana supervisionada** antes da geração de recortes anatómicos (*crops*) e matrizes de características.

Iniciamos com a verificação cruzada entre os ficheiros XML originais do LIDC-IDRI e as anotações da biblioteca `pylidc`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "pyproject.toml").is_file() and (p / "data").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start Jupyter from project/ or project/notebooks/")
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from IPython.display import display


## Verificação de Anotações LIDC e Preparação LNDb

<b>Propósito e Filtragem:</b><br>
Verificamos as anotações geradas pela biblioteca <code>pylidc</code> cruzando-as com o XML original descarregado do LIDC. A nossa tabela final reterá apenas a categoria <b>nódulo >= 3 mm</b>, garantindo foco no problema clínico principal e ignorando marcas irrelevantes ou micro-nódulos que acrescentariam ruído.
A verificação confirma que cada anotação esperada está presente(inclusão) e que a sua posição Z e os vértices do contorno (X/Y) coincidem exatamente. Isto fortifica a nossa pipeline contra erros de extração geométrica.

## 1. Verificação Rigorosa do XML e pylidc antes do Processamento

A extração de dados do LIDC-IDRI exige uma auditoria profunda entre os ficheiros XML originais (onde os médicos desenharam) e a base de dados gerada pela biblioteca pylidc. Esta extração realiza uma confirmação entre ambos os dados, de forma a não assumir nenhuma anotação.


<b>Auditoria de Filtragem do pylidc:</b><br>
Na auditoria da versão atual dos ficheiros XML, foram observados os seguintes casos, tratados durante a preparação das anotações:
<ul>
    <li><b>Registos Repetidos:</b> O XML local possui <b>170</b> registos de anotações duplicados ao longo de várias séries.</li>
    <li><b>Omissão de Dados Clínicos:</b> Duas marcas complexas de nódulos (<i>multi-edge-map</i>) não possuem quaisquer características associadas, sendo excluídas.</li>
    <li><b>Ruído de Desenho Médico:</b> O construtor omite contornos constituídos por apenas 1 ponto (<i>stray single-point marks</i>), que representam falhas ou cliques acidentais. Isto remove exatamente <b>64 contornos</b> distribuídos por 51 anotações.</li>
</ul>

Após a contabilização destas regras, garantimos que cada anotação elegível, classificação de malignidade e contorno retido *coincide perfeitamente* com o ficheiro original. O XML permanece estritamente como read-only, e as correspondências exatas fornecem a identidade da sessão de leitura para a nossa tabela de produção.

Para referência sobre o âmbito desta filtragem, consultar a [documentação de anotações do pylidc](https://pylidc.github.io/tuts/annotation.html).

In [2]:
from lung_project.lidc_xml import compare_xml_with_pylidc
from lung_project.lidc_pylidc import build_lidc_tables
import pandas as pd
from lung_project.paths import LIDC_ROOT, LNDB_ROOT, OUTPUT_ROOT
from lung_project.prepared import write_prepared, lidc_table, prepare_lndb

comparison = compare_xml_with_pylidc(LIDC_ROOT)
summary_keys = [
    "xml_files",
    "xml_multiedge_marks",
    "xml_duplicate_rows",
    "xml_duplicate_series",
    "xml_no_characteristics_rows",
    "xml_invalid_characteristics_rows",
    "xml_single_point_contours_omitted",
    "xml_annotations_affected_by_single_points",
    "xml_unique_eligible_annotations",
    "pylidc_annotations",
    "xml_contours_after_filter",
    "pylidc_contours",
    "xml_only_annotations",
    "pylidc_only_annotations",
    "contour_mismatches",
]
display(pd.DataFrame({"measure": summary_keys, "count": [comparison[key] for key in summary_keys]}))
print("XML entries without characteristics:", comparison["no_characteristics"])
print("Repeated XML examples:", comparison["duplicate_examples"])
assert (
    comparison["xml_multiedge_marks"]
    - comparison["xml_duplicate_rows"]
    - comparison["xml_no_characteristics_rows"]
    - comparison["xml_invalid_characteristics_rows"]
    == comparison["xml_unique_eligible_annotations"]
)
assert comparison["xml_unique_eligible_annotations"] == comparison["pylidc_annotations"]
assert comparison["xml_contours_after_filter"] == comparison["pylidc_contours"]
assert comparison["xml_only_annotations"] == comparison["pylidc_only_annotations"] == 0
assert comparison["contour_mismatches"] == 0
print("All eligible XML annotations, ratings, and retained contours match pylidc.")


/home/pera/aulas/31c/labiacd/lung/project/.venv/lib/python3.12/site-packages/pylidc/__init__.py:27: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as _pr


,measure,count
0,xml_files,1318
1,xml_multiedge_marks,7031
2,xml_duplicate_rows,170
3,xml_duplicate_series,6
4,xml_no_characteristics_rows,2
5,xml_invalid_characteristics_rows,0
6,xml_single_point_contours_omitted,64
7,xml_annotations_affected_by_single_points,51
8,xml_unique_eligible_annotations,6859
9,pylidc_annotations,6859


XML entries without characteristics: [('tcia-lidc-xml/187/044.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.111258527162678142285870245028', 'Nodule 001'), ('tcia-lidc-xml/187/191.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.109002525524522225658609808059', 'Nodule 002')]
Repeated XML examples: [('tcia-lidc-xml/188/083.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', '0'), ('tcia-lidc-xml/188/083.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', '157143'), ('tcia-lidc-xml/188/083.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', 'Nodule 001'), ('tcia-lidc-xml/188/083.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', '201255'), ('tcia-lidc-xml/188/084.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', '0'), ('tcia-lidc-xml/188/084.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001.153646219551578201092527860224', '157143'), ('tcia-lidc-xml/188/084.xml', '1.3.6.1.4.1.14519.5.2.1.6279.6001

## 2. Construção das tabelas de anotações e grupos candidatos a nódulo do LIDC

Para cada anotação (`pylidc.Annotation`), associamos a entrada correspondente do XML através da série, do identificador do nódulo, das classificações e da geometria dos contornos - incluindo a inclusão, a posição Z e as coordenadas dos vértices X/Y. Esta correspondência permite associar à anotação a sessão de leitura, o identificador do radiologista e a origem XML (`xml_source`). O `mark_id` é obtido da anotação `pylidc` e confirmado na correspondência com o XML. Se a identidade da sessão não puder ser determinada sem ambiguidade, o processo gera um erro em vez de atribuir uma sessão por suposição.

A função `Scan.cluster_annotations()` propõe agrupamentos espaciais de anotações; estes agrupamentos são propostas automáticas, não uma confirmação definitiva de que as anotações pertencem à mesma lesão.

<b>Sinalização de grupos para revisão:</b><br>
Um grupo é sinalizado como `ambiguous_group` se contiver mais de quatro anotações ou se incluir, na mesma série e sessão de leitura, anotações com `mark_id` diferentes. O segundo caso indica que a proposta espacial juntou marcas distintas do mesmo leitor.

<b>Ação:</b> Mantemos a proposta automática e assinalamos o grupo para revisão manual, em vez de fundir ou dividir as anotações automaticamente. Enquanto a revisão estiver pendente, o grupo não é utilizado para gerar um recorte CT; a pipeline só processa candidatos com estado `ready`.

No notebook 02, os recortes CT incluem uma margem de contexto de 6 mm por face, configurável por `contour_margin_mm` em `config/scan_processing.json`. A máscara corresponde à união dos contornos originais dos leitores retidos; a margem contextual fica fora da máscara. Os filtros podem usar o contexto CT, mas as estatísticas são calculadas apenas dentro dos contornos. Os contornos originais e as decisões humanas são preservados.

In [3]:
from lung_project.progress import notebook_progress

annotations, candidates = build_lidc_tables(progress=notebook_progress(), lidc_root=LIDC_ROOT)
automatic_candidates = candidates.copy()  # Preserve the original review inputs on reruns.
assert len(annotations) == comparison["pylidc_annotations"]
assert annotations.category.eq("nodule_ge_3mm").all()
assert candidates.annotation_count.sum() == len(annotations)
print("Reader annotations:", len(annotations), "patients:", annotations.patient_id.nunique())
print("Lesion groups:", len(candidates), "scans:", candidates.scan_uid.nunique())
print("Group statuses:", candidates.status.value_counts().to_dict())
display(annotations.head(5))
display(candidates.head(5))
print(
    "Pylidc proposals requiring membership review:",
    candidates.loc[candidates.grouping_conflict, "pylidc_candidate_id"].nunique(),
)
print(
    "Annotations in conflicting proposals:",
    int(candidates.loc[candidates.grouping_conflict, "annotation_count"].sum()),
)
assert candidates.loc[candidates.grouping_conflict, "status"].eq("ambiguous_group").all()


# Generate a browsable gallery of every automatic group and every contour slice.
from lung_project.lidc_review_visuals import write_lidc_gallery

contour_gallery = write_lidc_gallery(
    annotations,
    automatic_candidates,
    lidc_root=LIDC_ROOT,
    output_dir=OUTPUT_ROOT / "images" / "review" / "gallery",
    progress=notebook_progress(),
)
print("All-group contour gallery:", contour_gallery.resolve())


Matching pylidc annotations to XML reading sessions

pylidc: 883/883 annotated scans

Reader annotations: 6859 patients: 875
Lesion groups: 2651 scans: 883
Group statuses: {'ready': 2461, 'rating_needs_review': 170, 'ambiguous_group': 20}


,source_row,annotation_id,scan_uid,patient_id,mark_id,xml_source,reader_session,radiologist_id,category,contour_count,...,z_mm,subtlety_rating,internal_structure_rating,calcification_rating,sphericity_rating,margin_rating,lobulation_rating,spiculation_rating,texture_rating,malignancy_rating
0,0,4581,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,Nodule 001,tcia-lidc-xml/188/104.xml,1,423974236,nodule_ge_3mm,3,...,-298.115385,5,1,6,4,5,1,1,5,4
1,1,4582,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,Nodule 003,tcia-lidc-xml/188/104.xml,1,423974236,nodule_ge_3mm,2,...,-227.028571,4,1,6,5,5,1,1,5,3
2,2,4583,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,22134,tcia-lidc-xml/188/104.xml,2,349384903,nodule_ge_3mm,2,...,-226.945000,2,1,6,5,3,2,2,5,1
3,3,4584,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,22127,tcia-lidc-xml/188/104.xml,2,349384903,nodule_ge_3mm,3,...,-298.291176,3,1,6,4,4,1,1,5,2
4,4,4585,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,2148,tcia-lidc-xml/188/104.xml,3,581934703,nodule_ge_3mm,2,...,-226.916981,5,1,6,4,5,1,1,4,3


,source_row,scan_uid,patient_id,candidate_id,annotation_ids,source_rows,annotation_count,x_pixel,y_pixel,z_mm,status,quality_flags,grouping_method,grouping_policy,grouping_conflict,pylidc_candidate_id,pylidc_annotation_ids,reader_session_count
0,0,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,"[4581, 4584, 4586, 4587]","[0, 3, 5, 6]",4,45.036082,211.584046,-298.305336,ready,,pylidc.Scan.cluster_annotations,pylidc-session-conflicts-manual-review-v2,False,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,"[4581, 4584, 4586, 4587]",4
1,1,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,LIDC-IDRI-1001,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,"[4582, 4583, 4585]","[1, 2, 4]",3,405.697215,154.854784,-226.963518,ready,,pylidc.Scan.cluster_annotations,pylidc-session-conflicts-manual-review-v2,False,1.3.6.1.4.1.14519.5.2.1.6279.6001.100225287222...,"[4582, 4583, 4585]",3
2,2,1.3.6.1.4.1.14519.5.2.1.6279.6001.100332161840...,LIDC-IDRI-0778,1.3.6.1.4.1.14519.5.2.1.6279.6001.100332161840...,[5916],[7],1,313.625000,367.480769,-60.902451,ready,,pylidc.Scan.cluster_annotations,pylidc-session-conflicts-manual-review-v2,False,1.3.6.1.4.1.14519.5.2.1.6279.6001.100332161840...,[5916],1
3,3,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,LIDC-IDRI-0813,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,[5722],[8],1,163.484848,363.212121,866.972727,ready,,pylidc.Scan.cluster_annotations,pylidc-session-conflicts-manual-review-v2,False,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,[5722],1
4,4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,LIDC-IDRI-0813,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,"[5723, 5725, 5726]","[9, 11, 12]",3,384.460317,269.847619,876.657143,rating_needs_review,malignancy_amplitude_ge_3_agreement_lt_two_thirds,pylidc.Scan.cluster_annotations,pylidc-session-conflicts-manual-review-v2,False,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,"[5723, 5725, 5726]",3


Pylidc proposals requiring membership review: 20
Annotations in conflicting proposals: 101


Contour gallery: 2651/2651 groups

All-group contour gallery: /home/pera/aulas/31c/labiacd/lung/project/outputs/images/review/gallery/index.html


## 3. Avaliação de Malignidade e Definição da Variável Alvo (Target)
Para cada grupo de nódulos do pylidc, recolhemos as classificações de malignidade dadas pelos vários médicos. Estas são avaliações ordinais baseadas na opinião do médico, não sendo resultados confirmados.

<b>Cálculo do alvo:</b><br> Para cada grupo, calculamos a média das avaliações de malignidade e arredondamo-la ao inteiro mais próximo com a regra <code>floor(mean + 0.5)</code>. Assim, as metades exatas são arredondadas para cima (por exemplo, 1,5 passa a 2). Esta regra é definida explicitamente porque <code>round()</code> e <code>np.round()</code> usam, por omissão, o arredondamento para o par mais próximo (<i>ties-to-even</i>).

<b>Controlo de Ruído Médico (Amplitude \ge 3):</b><br> Avaliámos a métrica <code>max(ratings) - min(ratings)</code>. Anotações com amplitude >= 3 e menos de dois terços (2/3) dos médicos a atribuir exatamente a mesma nota ficam pendentes de uma <b>revisão manual</b>. O <i>target</i> continua a ser a média arredondada.

Com concordância exata >= 2/3, o grupo mantém-se elegível mesmo com amplitude >= 3, desde que o agrupamento esteja aceite. A concordância é a maior frequência de uma nota exata dividida pelo número de leitores; o alvo mantém-se a média arredondada, não a nota mais frequente.


In [4]:
from lung_project.labels import round_rating_mean
from lung_project.lidc_pylidc import build_lidc_malignancy_targets

targets = build_lidc_malignancy_targets(annotations, candidates)
assert len(targets) == len(candidates)
rating_review_required = targets.malignancy_amplitude.ge(3) & targets.malignancy_agreement.lt(2 / 3)
assert targets.malignancy_review_required.eq(rating_review_required).all()
assert targets.training_eligible.eq(
    targets.group_status.eq("ready") & ~rating_review_required
).all()
assert targets.loc[targets.training_eligible, "malignancy_target"].notna().all()
assert targets.loc[~targets.training_eligible, "malignancy_target"].isna().all()
assert targets.loc[targets.group_status.eq("ambiguous_group"), "target_status"].eq("group_needs_review").all()
assert targets.target_status.eq("rating_needs_review").eq(
    targets.group_status.eq("ready") & rating_review_required
).all()
assert targets.mean_rounded.eq(targets.malignancy_mean.map(round_rating_mean)).all()
ready_targets = targets.loc[targets.group_status.eq("ready")].copy()
print("Groups with accepted membership:", len(ready_targets))
print("Usable LIDC training targets:", int(targets.training_eligible.sum()))
print(
    "Groups needing rating review (amplitude >= 3 and exact agreement < 2/3):",
    int(targets.target_status.eq("rating_needs_review").sum()),
)
print("Groups needing membership review:", int(targets.group_status.eq("ambiguous_group").sum()))
print("Individual reader ratings:")
display(annotations.malignancy_rating.value_counts().sort_index().rename("annotations").to_frame())
print("Rounded lesion targets:")
display(
    targets.loc[targets.training_eligible, "malignancy_target"]
    .value_counts()
    .sort_index()
    .rename("lesions")
    .to_frame()
)


Groups with accepted membership: 2631
Usable LIDC training targets: 2461
Groups needing rating review (amplitude >= 3 and exact agreement < 2/3): 170
Groups needing membership review: 20
Individual reader ratings:


,annotations
malignancy_rating,
1,1020
2,1580
3,2606
4,962
5,691


Rounded lesion targets:


,lesions
malignancy_target,
1,286
2,598
3,1165
4,303
5,109


In [5]:
amplitude_counts = ready_targets.malignancy_amplitude.value_counts().sort_index().rename("lesions").to_frame()
amplitude_counts["percent"] = (100 * amplitude_counts.lesions / len(ready_targets)).round(1)
display(amplitude_counts.rename_axis("rating_amplitude"))

threshold_rows = []
for threshold in (2, 3):
    flagged = ready_targets.malignancy_amplitude.ge(threshold)
    multi_reader = ready_targets.reader_rating_count.ge(2)
    threshold_rows.append(
        {
            "amplitude_threshold": threshold,
            "flagged_groups": int(flagged.sum()),
            "flagged_percent_all_ready": round(100 * flagged.mean(), 1),
            "flagged_multi_reader_groups": int((flagged & multi_reader).sum()),
            "flagged_percent_multi_reader": round(100 * flagged[multi_reader].mean(), 1),
            "groups_below_threshold": int((~flagged).sum()),
        }
    )
display(pd.DataFrame(threshold_rows))

scenario_rows = []
for scenario, selected in [
    ("all ready groups", ready_targets),
    ("amplitude < 2", ready_targets.loc[ready_targets.malignancy_amplitude.lt(2)]),
    ("amplitude < 3", ready_targets.loc[ready_targets.malignancy_amplitude.lt(3)]),
]:
    rating_counts = selected.mean_rounded.value_counts()
    scenario_rows.append(
        {
            "scenario": scenario,
            "groups": len(selected),
            **{f"target_{rating}": int(rating_counts.get(rating, 0)) for rating in range(1, 6)},
        }
    )
print("Provisional rounded means under each candidate cutoff:")
display(pd.DataFrame(scenario_rows))
print("Selected training target status:")
display(targets.target_status.value_counts().rename("groups").to_frame())
rating_review = ready_targets.loc[
    ready_targets.target_status.eq("rating_needs_review"),
    [
        "candidate_id",
        "annotation_ids",
        "reader_rating_count",
        "malignancy_ratings",
        "malignancy_min",
        "malignancy_max",
        "malignancy_mean",
        "mean_rounded",
        "malignancy_amplitude",
        "malignancy_agreement",
    ],
].copy()
print("Rating review groups by rating span:")
display(rating_review.groupby(["malignancy_min", "malignancy_max"]).size().rename("groups").to_frame())
print("Examples awaiting rating review; all rows remain in rating_review and the saved CSV:")
display(rating_review.head(10))


,lesions,percent
rating_amplitude,,
0,1175,44.7
1,712,27.1
2,541,20.6
3,176,6.7
4,27,1.0


,amplitude_threshold,flagged_groups,flagged_percent_all_ready,flagged_multi_reader_groups,flagged_percent_multi_reader,groups_below_threshold
0,2,744,28.3,744,40.0,1887
1,3,203,7.7,203,10.9,2428


Provisional rounded means under each candidate cutoff:


,scenario,groups,target_1,target_2,target_3,target_4,target_5
0,all ready groups,2631,286,607,1247,382,109
1,amplitude < 2,1887,286,431,936,147,87
2,amplitude < 3,2428,286,582,1160,291,109


Selected training target status:


,groups
target_status,
available,2461
rating_needs_review,170
group_needs_review,20


Rating review groups by rating span:


groups
malignancy_min malignancy_max        
1              4                   49
               5                   21
2              5                  100

Examples awaiting rating review; all rows remain in rating_review and the saved CSV:


,candidate_id,annotation_ids,reader_rating_count,malignancy_ratings,malignancy_min,malignancy_max,malignancy_mean,mean_rounded,malignancy_amplitude,malignancy_agreement
4,1.3.6.1.4.1.14519.5.2.1.6279.6001.100398138793...,"[5723, 5725, 5726]",3,"[4, 3, 1]",1,4,2.666667,3,3,0.333333
6,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,"[4568, 4569, 4576, 4579]",4,"[5, 5, 3, 2]",2,5,3.750000,4,3,0.500000
7,1.3.6.1.4.1.14519.5.2.1.6279.6001.100621383016...,"[4567, 4570, 4575, 4580]",4,"[5, 5, 3, 2]",2,5,3.750000,4,3,0.500000
21,1.3.6.1.4.1.14519.5.2.1.6279.6001.102962801900...,"[1159, 1160, 1163, 1164]",4,"[5, 3, 3, 2]",2,5,3.250000,3,3,0.500000
25,1.3.6.1.4.1.14519.5.2.1.6279.6001.104562737760...,"[6042, 6043, 6044, 6045]",4,"[5, 5, 1, 1]",1,5,3.000000,3,4,0.500000
97,1.3.6.1.4.1.14519.5.2.1.6279.6001.112767175295...,"[1029, 1035, 1041, 1051]",4,"[2, 3, 5, 2]",2,5,3.000000,3,3,0.500000
114,1.3.6.1.4.1.14519.5.2.1.6279.6001.114218724025...,"[6309, 6312, 6314]",3,"[5, 3, 2]",2,5,3.333333,3,3,0.333333
124,1.3.6.1.4.1.14519.5.2.1.6279.6001.115386642382...,"[5246, 5247, 5250, 5251]",4,"[2, 2, 3, 5]",2,5,3.000000,3,3,0.500000
129,1.3.6.1.4.1.14519.5.2.1.6279.6001.115689755198...,"[79, 80, 82, 83]",4,"[2, 5, 3, 5]",2,5,3.750000,4,3,0.500000
142,1.3.6.1.4.1.14519.5.2.1.6279.6001.118140393257...,"[4745, 4748, 4752]",3,"[4, 1, 3]",1,4,2.666667,3,3,0.333333


## 4. Pausa para revisão manual no terminal

<b>Porque é necessária a revisão humana:</b><br>
A revisão permite inspecionar agrupamentos espaciais ambíguos e resolver casos de discordância elevada entre as avaliações de malignidade. As decisões são guardadas em `state/annotation_decisions.json`, associadas às tabelas de entrada, para que possam ser retomadas e auditadas. A revisão reduz o risco de erros, mas não garante, por si só, que todos os agrupamentos ou rótulos estejam corretos.

<b>Instruções de execução:</b><br>
Executar a célula abaixo. Quando terminar, interromper a execução do notebook, abrir um terminal na pasta raiz do projeto e executar:

```bash
uv run python -m lung_project review
```

In [6]:
# Save canonical label tables before interactive review. Rebuilding labels invalidates crops/features.
review_dir = PROJECT_ROOT / "state"
review_dir.mkdir(parents=True, exist_ok=True)
from lung_project.lidc_review import source_fingerprint, write_lidc_grouping

provenance = {
    "source_fingerprint": source_fingerprint(annotations, automatic_candidates),
    "grouping_policy": automatic_candidates.grouping_policy.iloc[0],
    "label_rounding": "nearest_integer_half_up",
}
reviewed_candidates, reviewed_targets, grouping_archive = write_lidc_grouping(
    annotations, automatic_candidates, prepared_dir=OUTPUT_ROOT / "prepared", review_dir=review_dir
)
if grouping_archive:
    print("Previous LIDC grouping and decisions archived at:", grouping_archive)
lndb_train, lndb_test = prepare_lndb(LNDB_ROOT)
# Preparation verifies the pinned publisher CSV and records both reader ID namespaces.
write_prepared("lndb_train", lndb_train, provenance=lndb_train.attrs["provenance"])
write_prepared("lndb_test", lndb_test, provenance=lndb_test.attrs["provenance"])
print("Saved the four canonical prepared tables. No separate review input snapshots are generated.")
print("Pause here. From project/, run: uv run python -m lung_project review")
print("Existing compatible decisions are replayed; continue with the next cell when review completes.")


Saved the four canonical prepared tables. No separate review input snapshots are generated.
Pause here. From project/, run: uv run python -m lung_project review
Existing compatible decisions are replayed; continue with the next cell when review completes.


### Carregar a revisão concluída e continuar

<b>Validação e aplicação das decisões:</b><br>
A célula abaixo carrega as tabelas de entrada preparadas e aplica-lhes as decisões guardadas em `state/annotation_decisions.json`. O registo tem de corresponder às entradas atuais; caso contrário, a validação interrompe o processo. Os <i>DataFrames</i> <code>candidates</code> e <code>targets</code> são atualizados na memória do notebook.

<b>Verificação de conclusão:</b><br>
Depois de aplicar as decisões, o código interrompe a execução se ainda houver grupos com revisão de agrupamento ou de malignidade pendente. Grupos removidos na revisão são mantidos como inelegíveis para treino.

<i>Nota analítica:</i> As estatísticas apresentadas antes desta etapa descrevem os agrupamentos automáticos originais do <i>pylidc</i>, antes da revisão manual. Na revisão do agrupamento, as 20 propostas ambíguas foram divididas em 59 grupos, pelo que o total passa de 2651 para 2690 grupos (2651 - 20 + 59). Destes, 125 foram removidos na revisão e 2565 ficam elegíveis para treino.

<b>Instruções:</b><br>
Executar esta célula depois de o comando de revisão no terminal indicar que a revisão terminou. Se a célula concluir sem erro, executar a célula seguinte para atualizar `outputs/prepared/lidc.csv` com os grupos, coordenadas e alvos revistos.

In [7]:
from lung_project.lidc_review import load_completed_review

candidates, targets = load_completed_review(review_dir, prepared_dir=OUTPUT_ROOT / "prepared")
assert not candidates.status.isin(["ambiguous_group", "rating_needs_review"]).any()
assert targets.training_eligible.eq(candidates.status.eq("ready")).all()
assert targets.loc[targets.training_eligible, "malignancy_target"].notna().all()
assert targets.loc[~targets.training_eligible, "malignancy_target"].isna().all()
print("Reviewed candidate statuses:", candidates.status.value_counts().to_dict())
print("Usable LIDC training targets:", int(targets.training_eligible.sum()))
display(
    targets.loc[targets.training_eligible, "malignancy_target"]
    .value_counts()
    .sort_index()
    .rename("lesions")
    .to_frame()
)


Reviewed candidate statuses: {'ready': 2565, 'excluded_by_review': 125}
Usable LIDC training targets: 2565


,lesions
malignancy_target,
1,289
2,604
3,1179
4,370
5,123


In [8]:
assert not candidates.status.isin(["ambiguous_group", "rating_needs_review"]).any(), (
    "Complete manual review first."
)
lidc = lidc_table(candidates, targets, automatic_candidates)
write_prepared("lidc", lidc, provenance=provenance)
print("Updated outputs/prepared/lidc.csv with reviewed membership, coordinates, and targets.")
print("LNDb label eligibility:")
display(lndb_train.target_status.value_counts().rename("findings").to_frame())
from lung_project.labels import characteristic_amplitude_summary

print("Reader characteristic amplitude counts (information only):")
print(characteristic_amplitude_summary({"lidc": lidc, "lndb_train": lndb_train}).to_string(index=False))


Updated outputs/prepared/lidc.csv with reviewed membership, coordinates, and targets.
LNDb label eligibility:


,findings
target_status,
ready,758
excluded_non_nodule,451
excluded_by_review,7
excluded_unknown_or_invalid_rating,3


Reader characteristic amplitude counts (information only):
   dataset     characteristic  nodules  known_amplitude  amplitude_ge_3
      lidc           subtlety     2690             2690             230
      lidc internal_structure     2690             2690              17
      lidc      calcification     2690             2690             144
      lidc         sphericity     2690             2690             119
      lidc             margin     2690             2690             258
      lidc         lobulation     2690             2690             226
      lidc        spiculation     2690             2690             194
      lidc            texture     2690             2690             141
lndb_train           subtlety      768              768               9
lndb_train internal_structure      768              768               4
lndb_train      calcification      768              768              20
lndb_train         sphericity      768              768              11
lndb_

<hr>

### Exportação e preparação do LNDb

As tabelas preparadas do projeto são guardadas em `outputs/prepared/`: `lidc_readers.csv`, `lidc.csv`, `lndb_train.csv` e `lndb_test.csv`. As tabelas do LIDC preservam os identificadores de candidatos e de anotações do `pylidc` para manter a rastreabilidade das correspondências e decisões.

<b>Verificação das anotações do LNDb:</b><br>
Na preparação do conjunto de treino LNDb, comparamos as anotações locais com uma cópia de referência das anotações dos autores. O ficheiro de referência é fixado por versão e o seu <i>checksum</i> SHA-256 é verificado. Para cada exame, o processo procura um mapeamento único entre os identificadores de leitores das duas fontes. A correspondência é validada pelas chaves das anotações e pelos valores de coordenadas, volume, indicador de nódulo e textura. Se não houver exatamente um mapeamento verificável, o processo assinala o exame como ambíguo ou interrompe a preparação, em vez de presumir a identidade dos leitores.

<b>Elegibilidade dos alvos:</b><br>
Os achados que não são nódulos, cuja correspondência entre leitores é ambígua ou que não têm avaliações de malignidade válidas permanecem na tabela para auditoria, mas não são elegíveis para treino. Nos achados LNDb com avaliações válidas, uma amplitude igual ou superior a 3, juntamente com uma concordância exata inferior a 2/3, resulta no estado <code>pending_rating_review</code>; estes casos são decididos no comando <code>uv run python -m lung_project review</code>, depois da revisão do LIDC, e as decisões ficam em <code>state/annotation_decisions.json</code>. Os sete casos atuais foram removidos (<code>excluded_by_review</code>) e continuam sem alvo de treino. Os restantes casos elegíveis recebem como alvo a média das avaliações arredondada ao inteiro mais próximo, com metades arredondadas para cima.

<b>Próximo passo:</b><br>
Com as tabelas preparadas e a proveniência registada, o projeto prossegue para o Notebook 02, onde serão processadas as imagens CT e gerados os recortes.